# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: Eric Ng Eng Chee



**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

# Libraries

In [ ]:
## Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

# 1. Business Understanding



**Problem.**
First time HDB buyers do not know what a fair price for HDB flats in Singapore are. This is because HDB prices can change a lot depending on town, flat type, floor area, which storey and how much lease is left. This leads to people possibly overpaying by thousands of dollars.

**Target Audience.**
First-time HDB buyers across Singapore who have shortlisted a flat and want a quick, data-based estimate before making an offer.


**Solution.**
A machine learning model that predicts a fair resale price for any HDB flat in Singapore, given its details (town, flat type, floor area, storey, lease years left, etc.). The model is then implemented as a Streamlit web app where user picks the flat's details from dropdowns and sliders, and the app returns an estimated fair price they can compare against the seller's asking price.



**Regression**

The target `resale_price` is a continuous dollar amount, not a category, so this is a regression problem. The output has to be usable directly as a dollar value.



**Dataset.**
Public HDB resale transaction data from **data.gov.sg**. Covers every completed resale from January 2017 onwards (~234,000 rows) across all 26 HDB towns in Singapore. Real, non-synthetic data.

# 2. Data Understanding

## 2.1 Load dataset

In [ ]:
## Read *.csv file into pandas DataFrame
FILE_PATH = "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv"
df = pd.read_csv(FILE_PATH)
df

## 2.2 Summary Statistics

In [ ]:
## Understand the type of variable for each column
df.info()

In [ ]:
## Check for missing data
df.isnull().sum()

In [ ]:
## Describe data distribution
df.describe(include="all")

In [ ]:
## Count unique values per column
df.nunique()

## 2.3 Data Visualization

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

In [ ]:
## Understanding distribution of target

# Histogram
df['resale_price'].hist(bins=50)
plt.xlabel('Resale Price (SGD)')
plt.ylabel('Frequency')
plt.title('Distribution of HDB Resale Prices')
plt.show()
## interpretation: The histogram shows that the resale price distribution is right-skewed, with most flats selling between ~$390k and ~$640k (the middle 50%). A long tail stretches past $1m, showing a small number of premium transactions well above the typical price.

# Boxplot
df['resale_price'].plot(kind='box')
plt.ylabel('Resale Price (SGD)')
plt.title('Boxplot of HDB Resale Prices')
plt.show()
## interpretation: The boxplot confirms the right skew, with a median resale price of ~$500k and many outliers sitting above the upper whisker (up to ~$1.73m). These outliers are real premium transactions (large flats, high-storey / DBSS units), not data errors, so they should be kept in the modelling data.

### 2.3.1.2 Understanding distribution of features

In [ ]:
## Understanding distribution of features

# Numeric features (histograms)
numeric_cols = ['floor_area_sqm', 'lease_commence_date']
df[numeric_cols].hist(bins=30, figsize=(12, 4))
plt.show()
## interpretation: The histograms show that floor_area_sqm is roughly bell-shaped and centered around 90-100 sqm (driven by the many 4-room and 5-room flats), while lease_commence_date spans 1966 to 2022 fairly evenly, meaning the dataset covers both old mature-estate flats and newer BTOs.

# Numeric features (boxplots)
df[numeric_cols].plot(kind='box', subplots=True, layout=(1, 2), figsize=(12, 4))
plt.show()
## interpretation: The boxplots show that floor_area_sqm has outliers on the high end (executive flats up to ~370 sqm) and a small number on the low end (1/2-room flats), while lease_commence_date is fairly symmetric with no strong outliers. The high-area outliers are genuine premium units, not data errors, so they should be kept in the modelling data.

# Categorical features (horizontal bar charts so long labels don't overlap)
categorical_cols = ['town', 'flat_type', 'flat_model', 'storey_range']
for col in categorical_cols:
    df[col].value_counts().sort_values().plot(kind='barh', figsize=(8, 6))
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.show()
## interpretation: The bar charts show that sales are heavily concentrated in a few categories: SENGKANG, PUNGGOL, WOODLANDS, TAMPINES and YISHUN dominate the towns while BUKIT TIMAH has only ~570 rows; 4 ROOM is by far the most common flat_type (~40% of rows) with 1 ROOM and MULTI-GENERATION having <100 rows each; 'Model A' and 'Improved' dominate flat_model; and most transactions are on floors 01-15 with very few above floor 30. The model will predict best for these common categories and less reliably for rare ones.

### 2.3.2 Understanding relationship between variables

In [ ]:
## Understanding relationship between variables

# Scatter: floor_area_sqm vs resale_price
plt.figure(figsize=(8, 5))
plt.scatter(df['floor_area_sqm'], df['resale_price'], alpha=0.1)
plt.xlabel('Floor Area (sqm)')
plt.ylabel('Resale Price (SGD)')
plt.title('Floor Area vs Resale Price')
plt.show()
## interpretation: The scatter plot shows a strong positive, roughly linear relationship between floor_area_sqm and resale_price — larger flats sell for more. Even a simple linear model should be able to capture most of this signal.

# Scatter: lease_commence_date vs resale_price
plt.figure(figsize=(8, 5))
plt.scatter(df['lease_commence_date'], df['resale_price'], alpha=0.1)
plt.xlabel('Lease Commence Year')
plt.ylabel('Resale Price (SGD)')
plt.title('Lease Commence Year vs Resale Price')
plt.show()
## interpretation: The scatter plot shows a positive but weaker relationship between lease_commence_date and resale_price. There is a large vertical spread at every year, meaning other factors (town, flat type, storey) also strongly influence price.

# Boxplot: town vs resale_price (horizontal so all 26 town labels stay readable)
plt.figure(figsize=(10, 8))
sns.boxplot(x='resale_price', y='town', data=df)
plt.title('Resale Price by Town')
plt.show()
## interpretation: The boxplot shows that town is a major driver of price. Central / mature towns (BUKIT TIMAH ~$777k, BISHAN ~$695k, QUEENSTOWN ~$670k, BUKIT MERAH ~$660k) have the highest medians, while outer towns (ANG MO KIO ~$418k, YISHUN ~$432k, BEDOK ~$433k) have the lowest — nearly a $360k median gap between the extremes. Town must stay as an input feature in the model.

# Boxplot: flat_type vs resale_price
plt.figure(figsize=(10, 5))
sns.boxplot(x='flat_type', y='resale_price', data=df)
plt.title('Resale Price by Flat Type')
plt.show()
## interpretation: The boxplot shows a clear step-up in resale price from 1 ROOM through EXECUTIVE — larger flat types cost more, as expected. MULTI-GENERATION flats also sit at the high end (they are essentially oversized executives). flat_type is clearly informative on its own.

# Boxplot: storey_range vs resale_price (horizontal, ordered from lowest floor to highest)
storey_order = sorted(df['storey_range'].unique())
plt.figure(figsize=(10, 7))
sns.boxplot(x='resale_price', y='storey_range', data=df, order=storey_order)
plt.title('Resale Price by Storey Range')
plt.show()
## interpretation: The boxplot shows a clear, near-monotonic rise in resale price as storey increases — the median goes from ~$450k on floors 01-03 up to ~$1.23m on floors 49-51. Storey is a strong ordinal feature, and its bucketed string form ("10 TO 12") should be turned into a numeric feature (e.g. the bucket midpoint) so the model can use the ordering.

# Boxplot: flat_model vs resale_price (horizontal, ordered by median price descending)
model_order = df.groupby('flat_model')['resale_price'].median().sort_values(ascending=False).index.tolist()
plt.figure(figsize=(10, 7))
sns.boxplot(x='resale_price', y='flat_model', data=df, order=model_order)
plt.title('Resale Price by Flat Model')
plt.show()
## interpretation: The boxplot shows a very wide price range across flat models. Premium models (Type S2 ~$1.13m, Type S1 ~$1.02m, Premium Apartment Loft ~$965k) command large premiums, while common models like Standard (~$345k), 2-room (~$365k) and New Generation (~$380k) sit at the low end. flat_model is a strong categorical feature and should be one-hot encoded so the model can price these levels separately.

# Line: median resale price by year (using the first 4 characters of month = YYYY)
df.groupby(df['month'].str[:4])['resale_price'].median().plot(figsize=(10, 4), marker='o')
plt.title('Median HDB Resale Price by Year')
plt.ylabel('Median Resale Price (SGD)')
plt.xlabel('Year')
plt.show()
## interpretation: The line chart shows the Singapore HDB market has trended sharply upwards over 2017-2026 — the median resale price rose from ~$410k in 2017 to ~$630k in 2026, a ~54% increase, with a clear acceleration from 2020 onwards. This confirms that a transaction-year feature is needed so the model can distinguish an older sale from a recent one at the same flat.

# Correlation heatmap of numeric variables
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()
## interpretation: The heatmap shows that floor_area_sqm is the strongest linear correlate with resale_price (r ~ 0.57), followed by lease_commence_date (r ~ 0.37). Pearson r only measures linear association, so tree-based models later may still find useful non-linear structure that a linear model misses.

# 3. Data Preparation

## 3.1 Data Cleaning

Before modelling, we make sure every row is clean and useful:

1. **Missing values** are checked again. Section 2.2 already showed there are none, so no imputation or row removal is needed.
2. **Duplicate rows** are checked. Any rows that are identical across every column are dropped so that each row counts as one unique transaction.
3. **ID-like columns** (`block` and `street_name`) are dropped. They have very high cardinality and act mainly as location identifiers, and `town` already captures location well enough for the model.

Note that we deliberately keep the high-price outliers seen during EDA, since those are real premium transactions. Removing them would make the model underprice large or high-floor flats.

In [ ]:
## Clean data

# Check for missing values (we saw 0 earlier, just confirming).
print("Total missing values:", df.isnull().sum().sum())

# Find and remove exact duplicate rows so each row is one unique sale.
print("Duplicate rows found:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows after cleaning:", df.duplicated().sum())

# Drop block and street_name. They have too many different values to be useful,
# and town already tells us the location.
df = df.drop(['block', 'street_name'], axis=1)

print("Shape after cleaning:", df.shape)
df.head()

In [ ]:
## Feature engineering
# Turn the text columns that really hold numbers into actual numbers.

# "61 years 04 months" -> 61.33 years
def lease_to_years(text):
    parts = text.split()
    years = int(parts[0])
    months = int(parts[2]) if len(parts) > 2 else 0   # some rows have no months
    return years + months / 12

df['remaining_lease_years'] = df['remaining_lease'].apply(lease_to_years)

# "10 TO 12" -> 11 (the middle floor)
def storey_to_mid(text):
    low, high = text.split(' TO ')
    return (int(low) + int(high)) // 2

df['storey_mid'] = df['storey_range'].apply(storey_to_mid)

# "2019-05" -> 2019 (the year the flat was sold)
df['txn_year'] = df['month'].str[:4].astype(int)

# Drop the old text columns now that we have number versions.
# Also drop lease_commence_date, because remaining_lease_years and txn_year
# already cover the same info (keeping all three confuses the linear model).
df = df.drop(['month', 'remaining_lease', 'storey_range', 'lease_commence_date'], axis=1)

print("Columns after feature engineering:", df.columns.tolist())
df.head()

In [ ]:
## One-hot encoding
# Models cannot read words, so we turn the text columns (town, flat_type,
# flat_model) into 0/1 columns. drop_first=True drops one column from each
# group to avoid repeating the same information.
df = pd.get_dummies(df, columns=['town', 'flat_type', 'flat_model'], drop_first=True)

print("Shape after encoding:", df.shape)
df.head()

## 3.2 Train-Test Split

We split the prepared data into a training set (used to fit the models) and a test set (kept unseen, used to measure how well the models predict on new data). We hold out 20% for testing and fix `random_state` so the split is reproducible and every model is judged on exactly the same rows.

In [ ]:
## Split data into train set and test set

# X = the inputs (everything except price). y = what we want to predict.
col_y = 'resale_price'
X = df.drop([col_y], axis=1)
y = df[col_y]

# Keep 80% for training and set aside 20% for testing.
# random_state makes the split the same every time we run it.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2026
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

# 4. Modelling

We train four different regression models and compare them on the same test set:

1. **Linear Regression** (baseline). A simple, fast starting point. Every other model has to beat this to be worth the extra complexity.
2. **Decision Tree**. Can capture non-linear patterns that a straight line cannot.
3. **Random Forest**. Many decision trees averaged together, usually more accurate and more stable than a single tree.
4. **Gradient Boosting**. Trees built one after another, where each new tree fixes the mistakes of the ones before it.

All four are scored on the same three metrics (RMSE, MAE and R2) so we can compare them fairly. We then pick the best one and tune it in the next section.

## 4.1 Baseline: Linear Regression

The simplest model we try. It fits a straight-line relationship between the features and price. Its scores become the baseline that every other model has to beat.

In [ ]:
## Initialise and train model

# Helper function so we train and score every model the same way.
# It also saves each model's scores into 'results' for a comparison table later.
results = []

def evaluate_model(model, name):
    model.fit(X_train, y_train)        # train on the training data
    y_pred = model.predict(X_test)     # predict on the unseen test data

    # Work out the three scores
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    print(name)
    print(f"  RMSE: ${rmse:,.0f}")
    print(f"  MAE : ${mae:,.0f}")
    print(f"  R2  : {r2:.4f}")
    return model

# Baseline: Linear Regression
linr = LinearRegression()
linr = evaluate_model(linr, "Linear Regression (baseline)")

## 4.2 Other models

Now we train the three tree-based models. Each one uses `random_state` so the results are the same every run. Random Forest also uses `n_jobs=-1`, which just means "use all CPU cores to train faster".

Note: Random Forest is the slowest to train (about 1 to 3 minutes on a normal laptop) because it builds many trees. This is normal, not a freeze.

In [ ]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=2026)
dt = evaluate_model(dt, "Decision Tree")

# Random Forest (n_jobs=-1 uses all CPU cores so it trains faster)
rf = RandomForestRegressor(random_state=2026, n_jobs=-1)
rf = evaluate_model(rf, "Random Forest")

# Gradient Boosting
gbr = GradientBoostingRegressor(random_state=2026)
gbr = evaluate_model(gbr, "Gradient Boosting")

## 4.3 Compare the models

We put all four models' scores into one table and sort by RMSE (lower is better) to see which one predicts prices most accurately.

In [ ]:
# Put all the model scores into one table, sorted by RMSE (lower = better)
results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
results_df

### Which model do we choose?

Sorting by RMSE, **Random Forest is the best model** by a clear margin:

| Model | RMSE | R2 |
| --- | --- | --- |
| Random Forest | ~$40,800 | 0.955 |
| Decision Tree | ~$52,000 | 0.927 |
| Gradient Boosting | ~$67,900 | 0.875 |
| Linear Regression (baseline) | ~$69,500 | 0.869 |

**Quantitative:** Random Forest cuts the average error almost in half compared to the baseline (RMSE from ~$69,500 down to ~$40,800) and explains about 95% of the price variation (R2 = 0.955).

**Qualitative:** Random Forest works well here because price depends on the features in a non-linear way (for example, storey and floor area interact with town). Averaging many trees captures those patterns while avoiding the overfitting that a single Decision Tree suffers from. Gradient Boosting does poorly with its default settings (barely better than the straight-line baseline), which is a good reason to try tuning, but Random Forest is already the strongest out of the box.

**Decision:** we choose **Random Forest** as our model and tune it in the next section to see if we can push the error down even further.


# 5. Model Evaluation

In [ ]:
## Evaluate model


In [ ]:
## New data

## Predict


## Iterative model development


In [ ]:
## Further feature engineering / feature selection